# PoofIt3D — reproducible Google Colab T4 setup

This notebook clones the existing repository branch, keeps the FastAPI/Diffusers and InstantMesh Python environments separate, runs a real sketch enhancement, then runs InstantMesh on an approved variation and validates the downloaded textured GLB. Colab `/content` files and installed packages disappear when a runtime resets; keep any user images you need in Drive. No secrets or API keys are stored here.

Before running, choose **Runtime → Change runtime type → T4 GPU**. Run the cells in order. The 3D cell performs real inference and may take several minutes.


In [ ]:
import subprocess, torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(gpu.total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import subprocess, sys

system_packages = ['git', 'ninja-build', 'ffmpeg', 'build-essential', 'libgl1', 'libglib2.0-0']
installed = subprocess.run(['dpkg-query', '-W', '-f=${Status}\n', *system_packages],
                           capture_output=True, text=True)
if installed.returncode != 0 or installed.stdout.count('install ok installed') != len(system_packages):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', *system_packages], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
print('System build and image libraries are ready.')


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/kakashsunny/PoofIt3D.git'
BRANCH = 'feat/instantmesh-sketch-to-3d'
REPO_ROOT = Path('/content/poofit3d-repository')
PROJECT = REPO_ROOT / 'PoofIt3D'
if not (REPO_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
if not (PROJECT / 'backend' / 'main.py').is_file():
    raise FileNotFoundError(f'Expected existing PoofIt3D app not found at {PROJECT}')
print('Project:', PROJECT)


In [ ]:
import subprocess, sys, torch

# Backend dependencies do not pin torch; keep Colab's CUDA build intact.
torch_before = torch.__version__
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(PROJECT / 'backend' / 'requirements.txt'),
                '-r', str(PROJECT / 'backend' / 'requirements-3d.txt')], check=True)
import torch as torch_after
print('Backend installed. PyTorch remains:', torch_after.__version__, '(before:', torch_before, ')')
assert torch_after.cuda.is_available(), 'CUDA is unavailable after installing backend requirements.'


In [ ]:
from pathlib import Path
import subprocess

INSTANTMESH_DIR = Path('/content/InstantMesh')
if not (INSTANTMESH_DIR / 'run.py').is_file():
    if INSTANTMESH_DIR.exists():
        raise RuntimeError(f'{INSTANTMESH_DIR} exists but is not an InstantMesh checkout; inspect it before replacing anything.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/TencentARC/InstantMesh.git', str(INSTANTMESH_DIR)], check=True)
if not (INSTANTMESH_DIR / 'configs' / 'instant-mesh-large.yaml').is_file():
    raise FileNotFoundError('Official InstantMesh config is missing from the checkout.')
print('InstantMesh checkout:', INSTANTMESH_DIR)


In [ ]:
import subprocess
from pathlib import Path

INSTANTMESH_ENV = Path('/content/instantmesh-venv')
INSTANTMESH_PYTHON = INSTANTMESH_ENV / 'bin' / 'python'
if not INSTANTMESH_PYTHON.exists():
    subprocess.run(['uv', 'venv', '--python', '3.10', '/content/instantmesh-venv'], check=True)
if not (INSTANTMESH_ENV / '.poofit3d-dependencies-installed').exists():
    def pip(args):
        subprocess.run(['uv', 'pip', 'install', '--python', str(INSTANTMESH_PYTHON), *args], check=True)
    pip(['--upgrade', 'pip', 'ninja'])
    pip(['torch==2.1.0', 'torchvision==0.16.0', 'torchaudio==2.1.0',
         '--index-url', 'https://download.pytorch.org/whl/cu121'])
    pip(['xformers==0.0.22.post7'])
    # Official requirements pin older Diffusers/Transformers; isolation prevents
    # them from downgrading the FastAPI environment.
    pip(['-r', str(INSTANTMESH_DIR / 'requirements.txt')])
    (INSTANTMESH_ENV / '.poofit3d-dependencies-installed').write_text('installed')
else:
    print('InstantMesh dependency marker found; skipping reinstall.')
subprocess.run([str(INSTANTMESH_PYTHON), '-c',
                'import torch, diffusers, transformers, rembg, trimesh; print(torch.__version__, diffusers.__version__, transformers.__version__); assert torch.cuda.is_available()'],
               cwd=INSTANTMESH_DIR, check=True)


In [ ]:
import os, subprocess
from pathlib import Path

ckpts = INSTANTMESH_DIR / 'ckpts'
ckpts.mkdir(exist_ok=True)
download = "from huggingface_hub import hf_hub_download; hf_hub_download('TencentARC/InstantMesh', 'diffusion_pytorch_model.bin', repo_type='model', local_dir=r'%s'); hf_hub_download('TencentARC/InstantMesh', 'instant_mesh_large.ckpt', repo_type='model', local_dir=r'%s')" % (ckpts, ckpts)
subprocess.run([str(INSTANTMESH_PYTHON), '-c', download], cwd=INSTANTMESH_DIR, check=True)
assert (ckpts / 'diffusion_pytorch_model.bin').is_file()
assert (ckpts / 'instant_mesh_large.ckpt').is_file()

BACKEND = PROJECT / 'backend'
ENV_FILE = BACKEND / '.env'
if not ENV_FILE.exists():
    ENV_FILE.write_text((BACKEND / '.env.example').read_text())
settings = {
    'HOST': '0.0.0.0', 'PORT': '8000', 'DEVICE': 'auto',
    'DEFAULT_MODEL_ID': 'controlnet_scribble', 'NUM_INFERENCE_STEPS': '20',
    'OUTPUT_IMAGE_SIZE': '512', 'DATASET_DIR': 'dataset',
    'INSTANTMESH_DIR': str(INSTANTMESH_DIR),
    'INSTANTMESH_PYTHON': str(INSTANTMESH_PYTHON),
    'INSTANTMESH_CONFIG': str(PROJECT / 'backend' / 'config' / 'instant-mesh-large-t4.yaml'),
    'INSTANTMESH_TIMEOUT': '1800',
}
lines = [line for line in ENV_FILE.read_text().splitlines() if not any(line.startswith(k + '=') for k in settings)]
lines.extend(f'{key}={value}' for key, value in settings.items())
ENV_FILE.write_text('\n'.join(lines) + '\n')
os.environ.update(settings)
print('InstantMesh checkpoints cached; T4 grid_res=64 config:', settings['INSTANTMESH_CONFIG'])


In [ ]:
import os, subprocess, sys, time, requests

API = 'http://127.0.0.1:8000/api'
try:
    health = requests.get(API + '/health', timeout=3)
    health.raise_for_status()
    print('Existing API:', health.json())
except requests.RequestException:
    log_path = '/content/poofit3d-fastapi.log'
    log = open(log_path, 'a')
    env = os.environ.copy()
    process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
                               cwd=BACKEND, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    Path('/content/poofit3d-fastapi.pid').write_text(str(process.pid))
    for _ in range(60):
        try:
            health = requests.get(API + '/health', timeout=2)
            if health.ok: break
        except requests.RequestException: pass
        time.sleep(2)
    else:
        print(Path(log_path).read_text()[-5000:])
        raise RuntimeError('FastAPI did not become healthy.')
print('Health:', health.status_code, health.json())
models = requests.get(API + '/models', timeout=20)
models.raise_for_status()
print('Models:', [item['id'] for item in models.json()])
assert 'controlnet_scribble' in [item['id'] for item in models.json()]


In [ ]:
import time, requests

# Real SD 1.5 + ControlNet Scribble smoke run on the repository's house sketch.
sketch = PROJECT / 'backend' / 'test_sketches' / 'house_sketch.png'
assert sketch.is_file(), f'Missing test sketch: {sketch}'
with sketch.open('rb') as image_file:
    response = requests.post(API + '/generate', files={'sketch_file': ('house_sketch.png', image_file, 'image/png')},
                             data={'style_id': 'photorealistic', 'model_id': 'controlnet_scribble',
                                   'description': 'small house, cream plaster walls, red tiled roof, brown wooden door, blue glass windows'},
                             timeout=60)
response.raise_for_status()
generation = response.json()
generation_id = generation['generation_id']
print('Generation:', generation)
for _ in range(1200):
    status = requests.get(API + f'/status/{generation_id}', timeout=15)
    status.raise_for_status()
    result = status.json()
    if result['status'] in ('completed', 'failed'): break
    time.sleep(3)
if result['status'] != 'completed':
    raise RuntimeError(f"Image generation failed or timed out: {result}")
print('Enhanced preview ready:', result['image_url'])


In [ ]:
import requests, trimesh
from pathlib import Path

# This is a notebook smoke-test approval of variation 0. In the product UI the
# user explicitly approves the selected preview before this API call is offered.
response = requests.post(API + '/3d/generate', data={'generation_id': generation_id, 'image_index': 0}, timeout=1900)
response.raise_for_status()
model = response.json()
assert model['status'] == 'completed' and model['textured'] is True
download_url = 'http://127.0.0.1:8000' + model['model_url']
asset = requests.get(download_url, timeout=120)
asset.raise_for_status()
glb_path = PROJECT / 'backend' / 'dataset' / 'result' / model['filename']
glb_path.write_bytes(asset.content)
scene = trimesh.load(glb_path, file_type='glb', force='scene')
assert scene.geometry, 'GLB has no geometry'
assert sum(len(mesh.faces) for mesh in scene.geometry.values()) > 0
print('InstantMesh result:', model)
print('Validated GLB:', glb_path, 'bytes:', glb_path.stat().st_size)


## Connect the React frontend

Keep FastAPI running. Expose port `8000` through an authenticated/temporary tunnel you control. Do not expose the unauthenticated prototype broadly. Build or deploy the existing `frontend/` with `REACT_APP_API_URL` set to the tunnel origin plus `/api` (for example `https://your-tunnel.example/api`). The React code centralizes all API, image, SSE, QR and GLB URLs through that setting. Set `FRONTEND_URL` to the frontend origin and `PUBLIC_URL` to the tunnel origin in `backend/.env`, then restart FastAPI for QR sharing and CORS. Never commit the temporary tunnel URL.

In the UI, generate images, choose a variation, approve it, then request InstantMesh. The notebook smoke test has already validated the API-produced GLB with trimesh. The frontend viewer uses `GLTFLoader`; orbit/zoom/pan and Reset camera are available after loading.

Colab runtimes are ephemeral. Store only source and important user assets in Drive; after every runtime reset, rerun setup and inference verification. The final result remains a single-image reconstruction, so hidden architecture is inferred rather than recovered exactly.
